In [1]:
!pip install -q --upgrade scanpy numpy scipy leidenalg igraph harmonypy cellrank 

In [ ]:
import scanpy as sc
import pandas as pd


from google.colab import drive

drive.mount('/content/drive')

data = sc.read_mtx('/content/drive/MyDrive/Genomics_iPSC/matrix.mtx.gz').T   # .T -> cells x genes

barcodes = pd.read_csv("/content/drive/MyDrive/Genomics_iPSC/barcodes.tsv.gz", sep='\t', header=None)
features = pd.read_csv("/content/drive/MyDrive/Genomics_iPSC/features.tsv.gz", sep='\t', header=None)

data.obs_names = barcodes[0].values
data.var_names = features[1].values   # column 1 = gene symbols (switch to [0] if head() shows symbols there)
data.var['gene_ids'] = features[0].values   # keep IDs too, handy later
data.var_names_make_unique()

print("cells x genes:", data.shape)

# DATA PROPROCESSING

1. QC Filtering 

First step in single cell RNA-seq analysis is to preprocess the data. I will start by filtering out genes and cells that don't match the criteria we want. This includes cells that don't express or over-express a certain number of genes, as well as genes that don't express themselves in a certain number of cells. 


In [ ]:
import scanpy as sc
import pandas as pd


# genes have to be expressed in at least 3 cells to be valid
sc.pp.filter_genes(data, min_cells = 3)

data.var['mt'] = data.var_names.str.startswith('mt-')

# Calculate Quality Control metrics
sc.pp.calculate_qc_metrics(data, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

# Filter data by QC metrics
data = data[data.obs['n_genes_by_counts'] > 200]
data = data[data.obs['n_genes_by_counts'] < 6000]
data = data[data.obs['total_counts'] < 50000]
data = data[data.obs['pct_counts_mt'] < 10]  


In [ ]:
# Biddy's dataset contains "timepoints" of when a cell was sequenced. Here we are adding a "timepoints" column into our data.
# I started with the suffix_order being in numerical order because that makes sense but then later after plotting realized "-2" comes before "-1"
data.obs['suffix'] = pd.Series(data.obs_names, index=data.obs_names).str.extract(r'(-\d+)$', expand=False).values

suffix_order = ['-2','-1','-3','-4','-5','-6','-7','-8']
data.obs['timepoint'] = pd.Categorical(data.obs['suffix'], categories=suffix_order, ordered=True)

print(data.obs['timepoint'].value_counts().sort_index())

# Since my macbook doesn't have enough RAM to run this project, I had to use colab to run it. I upload all the data to google drive to save the data
data.write("/content/drive/MyDrive/Genomics_iPSC/data_qc_filtered.h5ad")

2. Data Normalization

After QC filtering the data needs to be normalized and compressed. Normalizations takes each cell and normalized it so the raw counts sum to 10,000 (1e4). Without this a cell that was just sequenced more will appear to have higher gene expression than an identical cell that way just measured less. Log1p compresses the expression data so a few highly expressed genes don't dominate the whole pool. 

In [ ]:


import scipy.sparse as sp
import numpy as np, random

np.random.seed(0) # set the same random seed so that re-runs don't have wildly different results
random.seed(0)

data = sc.read_h5ad('/content/drive/MyDrive/Genomics_iPSC/data_qc_filtered.h5ad')

data.raw = data # save the raw date for use later

# Normalize and compress the data. This makes gene expression data per cell more accurate and accessible to our goal
sc.pp.normalize_total(data, target_sum=1e4)   
sc.pp.log1p(data)

3. Selecting Highly Variable Genes

We want genes that are variable. Genes that don't change expression level across out database are genes like robosomal proteins that have a baseline expression and don't contribute to our project. 

In [ ]:
# Selecting highly variable genes, select expression floor and ceiling and dispersion floor
sc.pp.highly_variable_genes(data, min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.pl.highly_variable_genes(data)

data = data[:, data.var['highly_variable']].copy()



4. Regressing and Scaling the data

Before we cluster the data, we need to make sure that nothing except gene expression controls how the cells cluster. "total_counts" and "pct_counts_mt" are unwanted sources of variation. Regressing the data takes the residual for each gene expressed across our cells and inputs that as out expression. Scaling takes the z_score for each gene: z = (value − mean) / standard_deviation. This allows a gene that is expressed more than average to have a positive z-score, less than average a negative score, and just average a 0. 

In [ ]:
# Regress the data. pp.regress_out fits a linear prediction based on given inputs trying to find how much of a genes expression is caused by those inputs. 
# Then it subtracts what those inputs contributed to the expression level from the expression to get the residual
sc.pp.regress_out(data, ['total_counts', 'pct_counts_mt'])

# Scale the data by computing z-score for each gene
sc.pp.scale(data, max_value=10)



5. Principal Component Analysis

So far all of our cells are expressed by 1800 data points, each being a different gene expression level per cell. A lot of this is just noise and running computations in a 1800 dimension space is very heavy, PCA reduces the dimensions by finding Principal Components (PC's) that can explain the expression level of many genes at once

In [ ]:

# Choose the top 50 PC's. This way you lose very little data
sc.pp.pca(data, n_comps=50, svd_solver='arpack', random_state = 0)
sc.pl.pca_overview(data)

# plot the variance ration in decending order with different genes
sc.pl.pca_variance_ratio(data, log=True)

suffix_order = ['-2','-1','-3','-4','-5','-6','-7','-8']
data.obs['timepoint'] = pd.Categorical(data.obs['suffix'], categories=suffix_order, ordered=True)

# color the pca graph with the timepoint also for better understanding
sc.pl.pca(data, color='timepoint')


6. Batch Correction

Biddys' experiment was ran twice. The first batch has a tag HF-1. The second batch has a tag HF-2. Ideally a fibroblast at the same timepoint in HF-1 and HF-2 should be the same, but they are off by a slight margin due to systemic or sequencing differences. Harmony can correct this so that clusters aren't driven by batch differences. 

In [ ]:

# Create a "replicates" column tracking HF-1 vs HF-2 for testing later
# Biddy's experiment was ran twice, first run is HF-1, second is HF-2. It makes sense to use one dataset as testing and one dataset as training
data.obs['replicate'] = pd.Series(data.obs_names, index=data.obs_names).str.extract(r'^(_HF-\d+_)', expand=False).values
print(data.obs['replicate'].value_counts())

# Save in google drive 
data.write('/content/drive/MyDrive/Genomics_iPSC/data_preprocessed.h5ad')

In [ ]:
import harmonypy
import numpy as np

data = sc.read_h5ad('/content/drive/MyDrive/Genomics_iPSC/data_preprocessed.h5ad')

# Runs harmony, which takes similar batches of cells in the PCA space and nudges them so they sit right on top of each other, thus erasing any batch difference that can contribute to cluster formation
harmony = harmonypy.run_harmony(data.obsm['X_pca'], data.obs, 'replicate', random_state=0)

# Check to see if harmony outouts the right shape, some formats do it differently (cells x pc's vs pc's x cells)
Z = np.asarray(harmony.Z_corr)
if Z.shape[0] != data.n_obs:        # orient to (cells, PCs) regardless of version
    Z = Z.T
data.obsm['X_pca_harmony'] = Z


# DATA MAPPING and ANNOTATION

1. Neighbors Graph and Clustering

This is where we start to answer the deeper biological question with "what clusters of similar gene expression can we find and how can we classify them"?. Start by building a graph, in this case a K_nearest_neighbors graph as we don't know how many clusters we'll be finding. Then run leiden, which is a clustering algorithm, to find clusters in the neighbors graph. Finallu we use UMAP to visually represent the 40 dimension neighbors graph in 2D so we can get a sense of understanding. 

In [ ]:


# Use 40 pc's from PCA and 15 neighbors to build a graph. K_nearest neighbors represents cells as nodes and edges link transcriptionally similar cells
sc.pp.neighbors(data, n_neighbors=15, n_pcs=40, use_rep='X_pca_harmony', random_state=0)

# Umap simply compresses the 40 dimensional graph into something visually readable to us in 2D. 
sc.tl.umap(data, random_state=0)

# the leiden algorithm actually defines the clusters, finding groups of densely connected nodes and adds them to "leiden harmony"
sc.tl.leiden(data, resolution=1.0, key_added='leiden_harmony', random_state=0)

# more UMAP visualizations for better context around the "replicate" and "timepoint" columns
sc.pl.umap(data, color=['replicate', 'leiden_harmony', 'timepoint'])
print(pd.crosstab(data.obs['leiden_harmony'], data.obs['replicate'], normalize='index'))

# Prints the leiden clusters overlapping the timepoint column. This is so we can see which clusters fall towards the beginning of sequencing (Fibroblast), and which fall towards the end (iEP/Dead-end)
print(pd.crosstab(data.obs['leiden_harmony'], data.obs['timepoint'], normalize='index'))

2. Ranking Gene Groups

This is where we start to annotate the clusters. Rank_gene_groups calculates the "marker genes", or which genes per cluster show unusually high expression level. pl.dotplot shows a dotplot of the gene expression level per cluster based on given merker genes. We decide these marker genes based on given biological data based on Biddy's data. pl.matrixplot is like the samw thing except we separate the genes into buckets of "fibroblast", "iEP", "dead-end", and "cycling". THese are the 4 possible states the cells can be in. 

In [ ]:
# tl.rank_)gene_groups uses the wilcoxon method, which pretty much compares the gene expression of each gene per cluster then takes the top "marker genes" per cluster 
sc.tl.rank_genes_groups(data, 'leiden_harmony', method='wilcoxon')
sc.pl.rank_genes_groups(data, n_genes=15, sharey=False)

# these are given markers from Biddy's paper that we use to show whether a cluster leans more one way or another
markers = ['Col1a1','Serpine1','Postn',             
           'Cdh1','Krt18','Krt19','Alb','Rbp1','Spint2',  
           'Mki67','Top2a',                          
           'FoxA1.HNF4a']                           

# Plot the clusters with darker and bigger dots showing more egen expression
sc.pl.dotplot(data, markers, groupby='leiden_harmony',
              use_raw=(data.raw is not None))



# same thing as the dotplot except now we've defined the different cell states and which genes go in each
marker_sets = {
    'Fibroblast': ['Col1a1','Serpine1','Postn'],
    'iEP':        ['Cdh1','Alb','Krt18','Krt19','Rbp1','Spint2'],
    'Dead-end':   ['Sfrp1','Dlk1','Peg3'],       
    'Cycling':    ['Mki67','Top2a'],
}

# pretty much a dotplot but more organized and shows which cell type each gene belongs in. Dendrogram creates a tree-like structure
sc.pl.matrixplot(data, marker_sets, groupby='leiden_harmony',
                 use_raw=(data.raw is not None),
                 standard_scale='var', cmap='Reds', dendrogram=True)

3. Annotate Clusters

This is one of the most important parts. Instead of manualy assigning a cluster to a cell type based on the matrixplot I wanted to create a function that could argmax the different gene expression levels and assign each cluster to a cell type based on which genes were the strongest.


In [ ]:
import numpy as np, pandas as pd
import scanpy as sc

# This is the only function in this project, mainly because it's more data handling then anything 
def auto_annotate(adata, marker_sets, cluster_key='leiden_harmony', use_raw=True):
    src = adata.raw if (use_raw and adata.raw is not None) else adata
    clusters = adata.obs[cluster_key].cat.categories # take the clusters and puts them into a dataframe as the rows
    scores = pd.DataFrame(index=clusters, columns=list(marker_sets), dtype=float) # columns are the labels: "Fibroblast", "iEP", etc

    for label, genes in marker_sets.items(): # This loop goes through each label 
        present = [g for g in genes if g in src.var_names]
        if not present:
            print(f"WARNING: no genes found for {label}")
            continue
        expr = src[:, present].X # take the expression matrix for each label
        expr = expr.toarray() if hasattr(expr, 'toarray') else np.asarray(expr)
        cell_score = expr.mean(axis=1).ravel() # save the mean of the expression levels for each label in the dataFrame
        # z-score across cells so marker sets of different magnitude compare fairly
        cell_score = (cell_score - cell_score.mean()) / (cell_score.std() + 1e-9) # Z-score the expression level. This is important because Fibroblast genes have baseline higher expression than something like iEP
        scores[label] = pd.Series(cell_score, index=adata.obs_names).groupby(
            adata.obs[cluster_key]).mean()

    assignment = scores.idxmax(axis=1)   # take the highest score for each cluster
    return assignment, scores

marker_sets = {
    'Fibroblast': ['Col1a1','Serpine1','Postn'],
    'iEP':        ['Cdh1','Alb','Krt18','Krt19','Rbp1','Spint2'],
    'Dead-end':   ['Sfrp1','Dlk1','Peg3'] 
}

# No cycling cells in marker_sets because we want cells that express similar expression acropss clusters as "transition", not cycling

assignment, scores = auto_annotate(data, marker_sets)
top = scores.max(axis=1) # Takes the max of the scores to see which label applies the best to each cluster

# This creates our transiton cells, where cells that score really low for the highest label or are very close between the first and second label are given "transition" as the label
margin = scores.apply(lambda r: r.nlargest(2).iloc[0] - r.nlargest(2).iloc[1], axis=1)
assignment[(top < 0.3) | (margin < 0.15)] = 'Transition'   # This is 
data.obs['cell_type'] = data.obs['leiden_harmony'].map(assignment).astype('category') # create a cell_type category with the different labels



print(data.obs['cell_type'].value_counts(dropna=False))   # confirm no NaN

# We need this umap to visualize the annotated clusters overlapped with the timepoint. This image helps give context to the next step: pseudotime
sc.pl.umap(data, color=['cell_type','timepoint'], legend_loc='on data')

data.write('/content/drive/MyDrive/Genomics_iPSC/data_annotated.h5ad')

4. Pseudotime

Our research question centers on if and how early we can predict fate labels from certain gene expression. Pseudotime helps us with the "how early" part by distinguishing cells by how much progress they've made along the trajectory, from Fibroblasts to iEP/Dead-end


In [ ]:
# PSEUDOTIME STEP
import scanpy as sc
import numpy as np

from google.colab import drive

drive.mount('/content/drive')

data_ps = sc.read_h5ad('/content/drive/MyDrive/Genomics_iPSC/data_annotated.h5ad')

# This computed the actual diffusion graph where pseudotime is calculated. Cells closer together in the graph space share more similar psueodtimes
sc.tl.diffmap(data_ps)

# we start with a root, calculated by taking one of the earliest cells, which will be a Fibroblast with time = -2
root_candidates = (data_ps.obs['cell_type'] == 'Fibroblast') & (data_ps.obs['timepoint'] == '-2')
data_ps.uns['iroot'] = np.flatnonzero(root_candidates.values)[0]

# actually run the pseudotime calculations, taking the root and calculating the distance to every other cell from there in the diffusion map 
sc.tl.dpt(data_ps)

# umaps purely for visualization
sc.pl.umap(data_ps, color=['dpt_pseudotime', 'timepoint', 'cell_type'])


5. PAGA Analysis

PAGA (partition based graph extraction) is a way to visualize how different groups of cells, in this case our cell types, are connected to each other. A strong Y-shape here indicates that the rajectory is flowing the way we want it to, from early Fibroblasts to Transition then distinguishing into either iEP or Dead-end cells. We run two PAGA analysis, one on the cell types and another on the individual clusters

In [ ]:
# run PAGA analisis

# Run PAGA on the cell type. PAGA uses the neighbor graph to se which cell types are near others, then constructs a graph. A thicker line means the groups are more connected``
sc.tl.paga(data_ps, groups="cell_type")

# Even though there are faint edges between groups like iEP and Dead-end, they are clearly weaker than the Y-shaped trajectory
sc.pl.paga(data_ps)

# same thing but with individual clusters. The output is messier but still points to the developmental trajectory we are looking for 
sc.tl.paga(data_ps, groups="leiden_harmony")
sc.pl.paga(data_ps, threshold=0.3, color='cell_type')


6. Cellrank Estimates

To build our classifier we need labels for our fibroblast cells to either be iEP or Deadend. This is probably the biggest limitation in my project, because Biddy's original paper used lineage tracking, where the decendants of Fibroblast cells were tracked to see if they became iEP or Deadend cells. However this involved a lot more data-parsing and preprocessing so I chose to cut it. Instead we use Cellrank, which uses transcriptionally similar cells to establish whether a Fibroblast cell will become iEP or Dead-end. This is less accurate than lineage tracking but computationally a lot more efficient

In [ ]:


from cellrank.kernels import PseudotimeKernel
from cellrank.estimators import GPCCA

# Take 10,000 samples and use the neighbors graph as the basis for Cellrank to calculate labels. We subsample because Cellrank is very memory heavy
data_sub = sc.pp.subsample(data_ps, n_obs=10000, copy=True)
sc.pp.neighbors(data_sub, n_neighbors=15, n_pcs=40, use_rep='X_pca_harmony')

# Cellrank computes the labels by modeling a random walk in a direction biased towards progression. The probability a walk will end up in a iEP or Deadend state is then calculated in the fate_probabilities

# This is why we needed pseudotime, the random walks move forward in progression, towards higher psuedotimes
kernel = PseudotimeKernel(data_sub, time_key="dpt_pseudotime")
kernel.compute_transition_matrix()

# GCCPA is the computational power behind Cellrank
g = GPCCA(kernel)

iep_cells = sorted(data_sub.obs_names[data_sub.obs['cell_type'] == 'iEP'])[:30]
de_cells  = sorted(data_sub.obs_names[data_sub.obs['cell_type'] == 'Dead-end'])[:30]

# Set the terminal states of "iEP" and "Dead_end"
g.set_terminal_states({"iEP": iep_cells, "Dead-end": de_cells})

# Calculate how likely a random walk will end up in iEP or Dead-end state
g.compute_fate_probabilities()
g.plot_fate_probabilities(basis="umap")
fp = g.fate_probabilities
fate_df = pd.DataFrame(fp, columns=list(fp.names), index=data_sub.obs_names)

# add to the object so it travels with the cells
data_sub.obs['p_iEP'] = fate_df['iEP'].values
data_sub.obs['p_deadend'] = fate_df['Dead-end'].values

# This prints essentially a matrix that tells us how many of each cell type was classified correctly
# One problem with Cellrank is that since iEP's are so much rarer than Dead-end cells, Cellrank will bias towards dead-end always as there are so many more dead-end cells
print(fate_df.groupby(data_sub.obs['cell_type']).mean())

# BUILDING CLASSIFIER

1. Data Validation Check

Before running the classifier, we should check if we have enough data and if that data is good enough to properly run a classifier. I run a couple checks to make sure the data isn't skewed at all towards one replicate or towards one label

In [ ]:
# 1. How many early cells survive the subset
early = data_sub.obs['cell_type'].isin(['Fibroblast','Transition']) & data_sub.obs['timepoint'].isin(['-2','-1'])
print(early.sum())

# cell type per replicate (HF-1 vs HF-2)
print(data_sub.obs.groupby('replicate')['cell_type'].value_counts())
subset = data_sub.obs['cell_type'].isin(['Fibroblast','Transition'])
print(data_sub.obs[subset].groupby('replicate')['cell_type'].value_counts())

# Check which Fibroblast/Transition cells Cellrank gives more than a 0.5 confidence score to 
subset = data_sub.obs['cell_type'].isin(['Fibroblast','Transition'])
sub = data_sub.obs[subset].copy()
sub['fate'] = (sub['p_deadend'] > 0.5).map({True:'deadend', False:'iEP'})
print(sub.groupby('replicate')['fate'].value_counts())

# Subset with a confidence filter
confident = subset & ((data_sub.obs['p_deadend'] > 0.7) | (data_sub.obs['p_deadend'] < 0.3))
sc = data_sub.obs[confident].copy()
sc['fate'] = (sc['p_deadend'] > 0.5).map({True:'deadend', False:'iEP'})
print(sc.groupby('replicate')['fate'].value_counts())

2. Classifier Preprocessing

Just like a regular classifier we have to preprocess the data. A key part of this is removing any genes that are directly associated with induced pluripotency/cycling. It's useless if the model returns genes that are the literal definition of induced pluripotency, and since cycling genes just add random noise we remove those as well

In [ ]:
# CLASSIFIER PREPROCESSING

import numpy as np

# all the following are Biddy's and just textbook cycling genes to remove
cycling = ['Mki67','Top2a','Kif11','Nusap1','Ckap2','Ckap2l','Cenpe','Cenpa','Cenpf',
        'Cdk1','Ccnb1','Ccnb2','Ccna2','Cdc20','Birc5','Bub1','Bub1b','Aurka','Aurkb',
        'Ube2c','Smc2','Smc4','Pcna','Rrm2','Tyms','Hmgb2','Hist1h1e','Tpx2','Prc1']

cc_present = [g for g in cycling if g in data_sub.var_names]
print(f"{len(cc_present)} cell-cycle genes matched:", cc_present)

# remove genes used to define fate as well as any cycling genes
genes_to_remove = ["Apoa1","Cdh1","Col1a2","Sfrp1"] + cc_present
keep_genes = data_sub.var_names.difference(genes_to_remove)
data_pre = data_sub[:, keep_genes].copy()

# FOr the training data only keep Fibroblast and Transition cells
data_pre = data_pre[data_pre.obs["cell_type"].isin(["Transition","Fibroblast"])].copy()

# split by replicate
train = data_pre[data_pre.obs["replicate"] == "_HF-1_"]
test  = data_pre[data_pre.obs["replicate"] == "_HF-2_"]

# all of the following is just regular sklearn preprocessing 
Xtr, Xte = train.X, test.X                     
ytr = (train.obs["p_deadend"] > 0.5).astype(int).values
yte = (test.obs["p_deadend"]  > 0.5).astype(int).values

print("train:", Xtr.shape, "class balance:", np.bincount(ytr))
print("test: ", Xte.shape, "class balance:", np.bincount(yte))


3. Logistic Regression Classifier 

The first of our classifications, we use logistic regression. We use three evaluation metrics: AUC, classification report, and confusion matrix. At the end we print the top predictive genes for each label, both for iEP and for dead-end fate 

In [ ]:


import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# class weight helps with the inbalance in the iEP vs dead-end data, otherwise the model would predict dead-end for everything
logreg = LogisticRegression(class_weight="balanced", max_iter=1000)
logreg.fit(Xtr, ytr)

# calculate both the probabilities the model gives to dead-end for AUC and hard label predictions for the classification report and confusion matrix
probabilities = logreg.predict_proba(Xte)[:, 1]     
predictions = logreg.predict(Xte)                

# Print our evaluations metrics 
print("AUC:", roc_auc_score(yte, probabilities))
print(classification_report(yte, predictions))
print(confusion_matrix(yte, predictions))

# print both the acensing and decending series so we get the top 15 predictive genes for each category
coef = pd.Series(logreg.coef_[0], index=data_pre.var_names)
print("\nTop dead-end predictors:\n", coef.sort_values(ascending=False).head(15))
print("\nTop iEP predictors:\n", coef.sort_values().head(15))


4. Random Forest Classifier

The second model I chose to run is a Random Forest Classifier. The reason I want to run two models is that while logistic regression is great at predicting linear trends. It assumes each genes contributes independently. However Random Forest builds many decision trees, and captures interactions between genes (does gene A matter only when gene B is high). If the AUC is similar for both then it means fate can be linearly predicted from gene expression

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# similar to logistic regression, balancing class weight and setting an arbitrary number of trees
forest = RandomForestClassifier(class_weight="balanced", n_estimators = 200, random_state=42)
forest.fit(Xtr, ytr)

proba = forest.predict_proba(Xte)[:, 1]     
preds = forest.predict(Xte)                 

# Same evaluations metrics
print("AUC:", roc_auc_score(yte, proba))
print(classification_report(yte, preds))
print(confusion_matrix(yte, preds))

# A key difference is that Random Forest outputs one list of top predictive genes, not two separate lists for top iEP vs top dead-end expression
features = pd.Series(forest.feature_importances_, index=data_pre.var_names)
print("Top predictive genes (random forest):\n", features.sort_values(ascending=False).head(20))



5. Predicting Fate from Timepoint

The original research question posited if fate could be predicted from gene expression and how early that fate could be predicted. The classifiers showed us that fate is genuinely predictable from expression of certain genes. Now it's time to find how early that fate can be predicted. To do this we take the weights from the logistic regression (no need for random forest as the AUC was virtually the same for both) and apply it to different timepoint bins to see if and how early the AUC rises from the earliest timepoints to the latest

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Split bins into different population controled bins. pd.qcut is better than pd.cut because it's better to have population controlled bins than width controlled bins
data_pre.obs["bin_data"] = pd.qcut(data_pre.obs["dpt_pseudotime"], q=16, labels=False)

# Build a loop which adds the auc, pseudotime, bin number, etc to a list so wde can graph the data
results = []
for b in sorted(data_pre.obs["bin_data"].unique()):
    cells = data_pre[data_pre.obs["bin_data"] == b]
    train = cells[cells.obs["replicate"] == "_HF-1_"]
    test  = cells[cells.obs["replicate"] == "_HF-2_"]

    ytr = (train.obs["p_deadend"] > 0.5).astype(int).values
    yte = (test.obs["p_deadend"]  > 0.5).astype(int).values

    # Make sure btoh labels (dead-end and iEP) are present in each bin 
    if len(np.unique(ytr)) < 2 or len(np.unique(yte)) < 2:
        print(f"bin {b}: skipped (one class)  n_test={len(yte)}")
        results.append({"bin": b, "median_pt": cells.obs["dpt_pseudotime"].median(),
                        "auc": None, "n_test": len(yte)})
        continue

    # Use the same weights as the Logistic Regression we ran before
    logreg = LogisticRegression(class_weight="balanced", max_iter=1000)
    logreg.fit(train.X, ytr)
    proba = logreg.predict_proba(test.X)[:, 1]

    results.append({"bin": b,
                    "median_pt": cells.obs["dpt_pseudotime"].median(),
                    "auc": roc_auc_score(yte, proba),
                    "n_test": len(yte),
                    "n_iep_test": int((yte == 0).sum())})

res = pd.DataFrame(results)
print(res)

# Graph the AUC vs paeudotime bins, see where the AUC rises significantly
valid = res.dropna(subset=["auc"])
ax = valid.plot.scatter(x="median_pt", y="auc")
ax.axhline(0.5, color="red", ls="--")   # chance reference
ax.set_xlabel("pseudotime (bin median)"); ax.set_ylabel("test AUC")

    
    